In [ ]:
!pip -q install openai requests beautifulsoup4

import json, re, getpass, requests
from bs4 import BeautifulSoup
from openai import OpenAI

api_key = getpass.getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)

MODEL = "gpt-5.4-mini"

memory = [{
    "role": "system",
    "content": (
        "You are a tiny research agent. "
        "Keep answers short, clear, and useful. "
        "If the user says they like short or detailed summaries, remember it."
    )
}]

prefs = {"summary_style": "short"}

def fetch_webpage(url: str, max_chars: int = 12000) -> str:
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
        tag.decompose()

    text = " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars] if text else "No useful content found."

tools = [{
    "type": "function",
    "name": "fetch_webpage",
    "description": "Fetch main visible text from a webpage URL",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "Full webpage URL"}
        },
        "required": ["url"],
        "additionalProperties": False
    }
}]

def update_pref(user_text: str):
    t = user_text.lower()
    if "short summaries" in t:
        prefs["summary_style"] = "short"
    elif "detailed summaries" in t:
        prefs["summary_style"] = "detailed"

def run_agent(user_text: str):
    update_pref(user_text)
    memory.append({
        "role": "user",
        "content": f"{user_text}\nUser summary preference: {prefs['summary_style']}."
    })

    response = client.responses.create(model=MODEL, input=memory, tools=tools)

    while True:
        calls = [x for x in response.output if x.type == "function_call"]

        if not calls:
            answer = response.output_text
            memory.append({"role": "assistant", "content": answer})
            return answer

        tool_outputs = []
        for call in calls:
            args = json.loads(call.arguments)
            result = fetch_webpage(**args)
            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": result
            })

        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs
        )

print(run_agent("Summarize this page in bullet points: https://en.wikipedia.org/wiki/Artificial_intelligence"))

Enter OpenAI API key: ··········
- Artificial intelligence (AI) is about building systems that can do tasks linked to human intelligence, like learning, reasoning, perception, and decision-making.
- It is used in things like search engines, chatbots, virtual assistants, autonomous vehicles, and game playing; generative AI can create text, images, audio, and video.
- AI research covers areas such as machine learning, natural language processing, computer vision, planning, and robotics.
- Common techniques include neural networks, statistics, optimization, logic, and search methods.
- The field began as an academic discipline in 1956 and has gone through cycles of excitement, setbacks, and “AI winters.”
- Interest rose sharply after 2012 with deep learning, and again after 2017 with transformers.
- Major challenges include reasoning, commonsense knowledge, uncertainty, and explainability.
- Ethical issues, safety, regulation, and possible long-term risks are major current concerns.



In [ ]:
!pip install feedparser

In [ ]:
import feedparser
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# -------- CONFIG --------
FEEDS = {
    "BBC": "http://feeds.bbci.co.uk/news/rss.xml",
    "Reuters": "https://feeds.reuters.com/Reuters/worldNews",
    "TechCrunch": "https://techcrunch.com/feed/",
}

EMAIL_USER = "21a51a05h1@adityatekkali.edu.in"
EMAIL_PASS = "elmk xdph tndi kdms"   # Use Gmail App Password
EMAIL_TO = "21a51a05h2@adityatekkali.edu.in"
# ------------------------


def fetch_news():
    articles = []
    seen = set()

    for source, url in FEEDS.items():
        feed = feedparser.parse(url)
        for entry in feed.entries[:5]:
            title = entry.title
            link = entry.link

            if title.lower() in seen:
                continue
            seen.add(title.lower())

            articles.append((source, title, link))

    return articles


def create_email_content(articles):
    html = "<h2>Daily News Update</h2><ul>"
    for source, title, link in articles:
        html += f"<li><b>{source}</b>: {title}<br><a href='{link}'>Read more</a></li><br>"
    html += "</ul>"
    return html


def send_email(content):
    msg = MIMEMultipart()
    msg["From"] = EMAIL_USER
    msg["To"] = EMAIL_TO
    msg["Subject"] = "Daily News Digest"

    msg.attach(MIMEText(content, "html"))

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(EMAIL_USER, EMAIL_PASS)
        server.send_message(msg)



news = fetch_news()
email_content = create_email_content(news)
send_email(email_content)

print("✅ Email sent successfully!")

✅ Email sent successfully!
